In [1]:
#All packages needed to run TwINFER simulation and inference are listed here. 
#If any of them are not installed, please install them using pip or conda env.
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numba
import tqdm
import scipy
import seaborn
import os
import sys
import joblib
from itertools import product
from pathlib import Path
import json

In [2]:
path_to_simulations = "/projects/b1042/GoyalLab/jaekj/TWINFER/TwINFER/grnInference/simulation_data/test/figure_3_simulations/"
path_to_code_repo = "/home/gzu5140/Keerthana_b1042/grnInference/code/TwINFER/"
path_to_plot_data = "/home/gzu5140/Keerthana_b1042/grnInference/plot_data/figure_3_28012026/"
os.makedirs(path_to_plot_data, exist_ok=True)

## Importing necessary functions and defining functions used

In [6]:
# Calculation functions
import sys
sys.path.append(str(path_to_code_repo))
import importlib
from TwINFER_function_scripts import correlation_analysis_functions
from TwINFER_function_scripts import correlation_analysis_helpers
from TwINFER_function_scripts import infer_with_twinfer

importlib.reload(correlation_analysis_functions)
importlib.reload(correlation_analysis_helpers)
importlib.reload(infer_with_twinfer)

from TwINFER_function_scripts.correlation_analysis_functions import (
    generate_random_shuffle
    # calculate_pairwise_gene_gene_correlation_matrix,
    # check_system_in_steady_state,
    # check_gene_gene_correlation_threshold,
    # calculate_twin_random_pair_correlations,
    # differentiate_single_state_reg_and_multiple_states,
    # identify_reg_if_multiple_states,
    # get_cross_correlations,
    # identify_actual_directed_edges
)

# Helper functions
from TwINFER_function_scripts.correlation_analysis_helpers import (
    extract_param_index,
    read_input_matrix,
    split_and_merge_simulations,
    get_param_data, 
    plot_matrix_as_heatmap,
    print_summary,
    plot_network
)

from TwINFER_function_scripts.infer_with_twinfer import (
    infer_with_twinfer
)

In [7]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed
from scipy.stats import spearmanr

#Functions to calculate cross-correlations for multiple replicates. Same idea as step 3 of infer_with_twinfer pipeline - rewritten here for optimizing the input
# and output for this specific use-case

# ==========================================
# CORRELATION FUNCTIONS
# ==========================================
def compute_correlation_matrix_fast(gene_matrix_t1, gene_matrix_t2, gene_list, 
                                   gene_pairs=None, threshold=None):
    """Fast correlation computation using pure NumPy operations."""
    n_genes = len(gene_list)
    gene_to_idx = {gene: i for i, gene in enumerate(gene_list)}
    
    # Initialize matrices
    raw_matrix = np.zeros((n_genes, n_genes))
    
    # Determine which pairs to compute
    if gene_pairs is None:
        pairs_to_compute = [(i, j) for i in range(n_genes) for j in range(n_genes)]
    else:
        pairs_to_compute = []
        for gene_1, gene_2 in gene_pairs:
            if gene_1 in gene_to_idx and gene_2 in gene_to_idx:
                i, j = gene_to_idx[gene_1], gene_to_idx[gene_2]
                pairs_to_compute.append((i, j))
    
    # Compute correlations for specified pairs
    for i, j in pairs_to_compute:
        corr = spearmanr(gene_matrix_t1[i, :], gene_matrix_t2[j, :]).correlation
        if np.isnan(corr):
            corr = 0.0
        raw_matrix[i, j] = corr
    
    # Convert to DataFrames
    raw_df = pd.DataFrame(raw_matrix, index=gene_list, columns=gene_list)
    return {"raw_matrix": raw_df}

def single_cell_shuffle(gene_matrix_t1, gene_matrix_t2, gene_list, gene_pairs, threshold):
    """Perform a single cell shuffling iteration."""
    n_cells = gene_matrix_t1.shape[1]
    shuffled_indices = np.random.permutation(n_cells)
    shuffled_matrix_t2 = gene_matrix_t2[:, shuffled_indices]
    return compute_correlation_matrix_fast(gene_matrix_t1, shuffled_matrix_t2, gene_list, gene_pairs, threshold)

def prepare_twin_data(df, gene_list, t1=10, t2=20):
    """
    Prepare twin data from simulation CSV file.
    Based on your data loading process.
    """
    # Basic checks
    if 'clone_id' not in df.columns:
        raise ValueError("CSV must contain 'clone_id' column")
    if 'time_step' not in df.columns:
        raise ValueError("CSV must contain 'time_step' column")
    
    n_clones_simulation = df['clone_id'].nunique()
    
    # Shuffle all clone IDs (like in your code)
    np.random.seed(101010)
    clone_ids_shuffled = np.random.permutation(n_clones_simulation)
    
    # Split into ratios (like in your code)
    n1 = n2 = n_clones_simulation // 4
    across_t_clones = clone_ids_shuffled[n1 + n2:]
    
    # Get across-time twins (like in your code)
    across_t_twin1 = df[
        (df['clone_id'].isin(across_t_clones)) & 
        (df['time_step'] == t1) & 
        (df['replicate'] == 1)
    ].reset_index(drop=True)
    
    across_t_twin2 = df[
        (df['clone_id'].isin(across_t_clones)) & 
        (df['time_step'] == t2) & 
        (df['replicate'] == 2)
    ].reset_index(drop=True)
    
    return across_t_twin1, across_t_twin2

def analyze_single_file(file_path, gene_list, t1=1, t2=20, n_shuffles=10000, threshold=0.02):
    """
    Analyze a single CSV file to get correlations and thresholds.
    """
    
    print(f"  Analyzing {os.path.basename(file_path)}")
    
    # Load data
    df = pd.read_csv(file_path)
    
    # Prepare twin data
    rep_0_t1, rep_1_t2 = prepare_twin_data(df, gene_list, t1, t2)
    
    if len(rep_0_t1) == 0 or len(rep_1_t2) == 0:
        print(f"    Warning: No twin data found")
        return None
    
    # Extract gene matrices
    gene_matrix_t1 = []
    gene_matrix_t2 = []
    
    for gene in gene_list:
        # Look for gene columns
        gene_col_t1 = f"{gene}_mRNA" if f"{gene}_mRNA" in rep_0_t1.columns else None
        gene_col_t2 = f"{gene}_mRNA" if f"{gene}_mRNA" in rep_1_t2.columns else None
        
        if not gene_col_t1:
            matching_cols = [col for col in rep_0_t1.columns if gene in col and 'mRNA' in col]
            gene_col_t1 = matching_cols[0] if matching_cols else None
            
        if not gene_col_t2:
            matching_cols = [col for col in rep_1_t2.columns if gene in col and 'mRNA' in col]
            gene_col_t2 = matching_cols[0] if matching_cols else None
        
        if gene_col_t1 and gene_col_t2:
            gene_matrix_t1.append(rep_0_t1[gene_col_t1].values)
            gene_matrix_t2.append(rep_1_t2[gene_col_t2].values)
        else:
            print(f"    Warning: Could not find {gene} data")
            return None
    
    gene_matrix_t1 = np.array(gene_matrix_t1)
    gene_matrix_t2 = np.array(gene_matrix_t2)
    
    gene_pairs = [('gene_1', 'gene_2'), ('gene_2', 'gene_1'), ('gene_1', 'gene_1'), ('gene_2', 'gene_2')]
    
    # Get actual correlations
    actual_results = compute_correlation_matrix_fast(gene_matrix_t1, gene_matrix_t2, gene_list, gene_pairs, None)
    actual_matrix = actual_results['raw_matrix']
    
    gene_1_to_2 = actual_matrix.loc['gene_1', 'gene_2']
    gene_2_to_1 = actual_matrix.loc['gene_2', 'gene_1']
    
    # Get shuffled correlations for thresholds
    shuffled_results = Parallel(n_jobs=-1, verbose=0)(
        delayed(single_cell_shuffle)(gene_matrix_t1, gene_matrix_t2, gene_list, gene_pairs, None)
        for _ in range(n_shuffles)
    )
    
    # Calculate thresholds
    shuffled_12 = []
    shuffled_21 = []
    
    for result in shuffled_results:
        shuffled_matrix = result['raw_matrix']
        shuffled_12.append((shuffled_matrix.loc['gene_1', 'gene_2']))
        shuffled_21.append((shuffled_matrix.loc['gene_2', 'gene_1']))

    # Calculate threshold for this pair: gene 1 -> gene 2
    p_plus_12 = np.mean(shuffled_12 >= gene_1_to_2)
    p_minus_12 = np.mean(shuffled_12 <= gene_1_to_2)
    p_value_12 = min(2 * p_plus_12, 2 * p_minus_12, 1.0)
    is_significant_12 = p_value_12 < threshold
    print(f"Observed correlation: {gene_1_to_2:.4f}")
    print(f"p-value: {p_value_12:.4f}")
    print(f"Significant at α={threshold}: {is_significant_12}")

    # Calculate threshold for this pair: gene 1 -> gene 2
    p_plus_21 = np.mean(shuffled_21 >= gene_2_to_1)
    p_minus_21 = np.mean(shuffled_21 <= gene_2_to_1)
    p_value_21 = min(2 * p_plus_21, 2 * p_minus_21, 1.0)
    is_significant_21 = p_value_21 < threshold
    print(f"Observed correlation: {gene_2_to_1:.4f}")
    print(f"p-value: {p_value_21:.4f}")
    print(f"Significant at α={threshold}: {is_significant_21}")
    
    threshold_12 = max(
        abs(np.percentile(shuffled_12, 1)),
        abs(np.percentile(shuffled_12, 99))
    )
    threshold_21 = max(
        abs(np.percentile(shuffled_21, 1)),
        abs(np.percentile(shuffled_21, 99))
    )

    combined_null = np.concatenate([shuffled_12, shuffled_21])
    threshold_combined = max(
        abs(np.percentile(combined_null, 1)),
        abs(np.percentile(combined_null, 99))
    )

    
    print(f"    Correlations: gene_1→gene_2={gene_1_to_2:.3f}, gene_2→gene_1={gene_2_to_1:.3f}")
    print(f"    Thresholds: gene_1→gene_2={threshold_12:.3f}, gene_2→gene_1={threshold_21:.3f}")
    
    return {
        'gene_1_to_gene_2': gene_1_to_2,
        'gene_2_to_gene_1': gene_2_to_1,
        'pvalue_12': p_value_12,
        'pvalue_21': p_value_21,
        'threshold_12': threshold_12,
        'threshold_21': threshold_21,
        'threshold_combined': threshold_combined
    }
# =========================================================================================================

def cross_correlation_data_across_replicates(base_path, folder_names, gene_list, t1=1, t2=20, n_shuffles=10000):
    """
    Collect correlations and thresholds from all files specified as this specific network type.
    """
    
    all_results = []
    
    for folder in folder_names:
        folder_path = os.path.join(base_path, folder)
        print(f"Processing {folder}...")
        
        if not os.path.exists(folder_path):
            print(f"  Folder not found: {folder_path}")
            continue
            
        # Find CSV files
        csv_files = [f for f in os.listdir(folder_path) if f.startswith('df_') and f.endswith('.csv') and ("0801" in f or "0901" in f)][:20]
        print(f"  Found {len(csv_files)} files")
        for csv_file in csv_files:
            file_path = os.path.join(folder_path, csv_file)
            print(csv_file)
            try:
                result = analyze_single_file(file_path, gene_list, t1, t2, n_shuffles)
                
                if result:
                    result['Condition'] = folder
                    result['File'] = csv_file
                    all_results.append(result)
                    
            except Exception as e:
                print(f"    Error with {csv_file}: {e}")
                continue
    
    return pd.DataFrame(all_results)

In [5]:
# Network types (folder names)
network_types = [
    "A_B",
    "A_rep_B", 
    "A_and_B_both_repress",
    "A_rep_B_B_to_A",
    "A_to_B",
    "A_and_B"
]

# Base configuration template
base_config_template = {
    'n_cells': 6000,
    'simulation_time_before_division': 1000,
    'twin_simulation_time_after_division': 48,
    'twin_measurement_resolution': 1,
    "path_to_connectivity_matrix": f"{path_to_code_repo}/simulation_example_input_data/connectivity_matrix_A_to_B.txt",  # Will be updated per network type
    "param_csv": f"{path_to_code_repo}/simulation_example_input_data/median_parameter.csv",  # Will be updated per network type
    "rows_to_use": [[0,1]],
    "path_to_plot_data": path_to_plot_data,
    "log_file": None,  # Will be updated per network type
    "type": None,  # Will be updated per network type
    "number_of_parallel_parameters": 1,
    "number_of_cores_per_parameter": 18,
}

# Time points for analysis in figure 3
t1, t2 = 1, 20  

In [ ]:
# Process each network type
all_correlation_matrices = {}

for network_type in network_types:
    print(f"Processing network type: {network_type}")
    
    # Find simulation file for this network type
    network_folder = os.path.join(path_to_simulations, network_type)
    if not os.path.exists(network_folder):
        print(f"Warning: Network folder not found: {network_folder}")
        continue
    
    # Look for CSV files in the network type folder
    csv_files = [f for f in Path(network_folder).glob("*.csv")]
    print(csv_files)
    if not csv_files:
        print(f"Warning: No CSV files starting with 'df' found in {network_folder}")
        continue
    
    # Use the first CSV file found (or you can add logic to select specific one)
    path_to_simulation_file = str(csv_files[0])
    print(f"Using simulation file: {os.path.basename(path_to_simulation_file)}")
    
    # Update config for this network type
    config = base_config_template.copy()
    config["type"] = network_type
    config["log_file"] = f"{path_to_code_repo}/example_simulation_output/{network_type}_log.jsonl"
    
    # Check if required files exist
    if not os.path.exists(config["path_to_connectivity_matrix"]):
        print(f"Warning: Connectivity matrix not found for {network_type}")
        continue
    print(config["param_csv"])
    if not os.path.exists(config["param_csv"]):
        print(f"Warning: Parameter CSV not found for {network_type}")
        continue

    try:
        # Run inference for this network type
        correlation_matrices = infer_with_twinfer(
                path_to_simulation_file = path_to_simulation_file, 
                base_config = config, 
                t1 = t1, t2 = t2,
                check_for_steady_state=False, 
                plot_correlation_matrices_as_heatmap=False, 
                have_any_output=False,
                infer_direction_for_which_edges = "all-edges",
                merge_to_multiple_states  = False,
                match_sim_details = False
            )
            
        # Store the correlation matrices
        all_correlation_matrices[network_type] = correlation_matrices
            
        # Save the directional correlation matrix for this network type
        correlation_file_name = f"{path_to_plot_data}/filtered_directional_correlation_type_{network_type}.csv"
        correlation_matrices['direction_matrix'].to_csv(correlation_file_name)            
        print(f"Successfully processed {network_type}")
        print(f"Saved correlation matrix to: {correlation_file_name}")
    except Exception as e:
        print(f"Error processing {network_type}: {str(e)}")
        continue

### Code for calculating cross-correlations across all replicates

In [15]:
# Parameters
sub_folder_names = ["A_B", "A_to_B", "A_and_B", "A_rep_B", "A_and_B_both_repress", "A_rep_B_B_to_A", ]
gene_list = ['gene_1', 'gene_2']
t1, t2 = 1, 20  # Adjust these to your time points
n_shuffles = 10000  # Number of shuffles for threshold calculation

# Collect all data
print("\n1. Collecting data from all files...")
df_results = cross_correlation_data_across_replicates(path_to_simulations, sub_folder_names, gene_list, t1, t2, n_shuffles)

if df_results.empty:
    print("No data collected!")
else:
    df_results.to_csv(f"{path_to_plot_data}/box_plot_data.csv")
print(f"Collected {len(df_results)} files from {df_results['Condition'].nunique()} conditions")

# Summary stats
print("\n4. Summary statistics:")
summary = df_results.groupby('Condition')[['gene_1_to_gene_2', 'gene_2_to_gene_1']].describe()
print(summary)


1. Collecting data from all files...
Processing A_B...
  Found 0 files
Processing A_to_B...
  Found 0 files
Processing A_and_B...
  Found 10 files
df_rows_0_0_08012026_224459_ncells_6000_A_and_B_rep_8_abf5a605.csv
  Analyzing df_rows_0_0_08012026_224459_ncells_6000_A_and_B_rep_8_abf5a605.csv
Observed correlation: 0.0337
p-value: 0.0548
Significant at α=0.02: False
Observed correlation: 0.1080
p-value: 0.0000
Significant at α=0.02: True
    Correlations: gene_1→gene_2=0.034, gene_2→gene_1=0.108
    Thresholds: gene_1→gene_2=0.042, gene_2→gene_1=0.043
df_rows_0_0_08012026_211444_ncells_6000_A_and_B_rep_5_f5055d94.csv
  Analyzing df_rows_0_0_08012026_211444_ncells_6000_A_and_B_rep_5_f5055d94.csv
Observed correlation: 0.0757
p-value: 0.0002
Significant at α=0.02: True
Observed correlation: 0.0852
p-value: 0.0000
Significant at α=0.02: True
    Correlations: gene_1→gene_2=0.076, gene_2→gene_1=0.085
    Thresholds: gene_1→gene_2=0.044, gene_2→gene_1=0.044
df_rows_0_0_08012026_231506_ncells_

## 5-gene cascade

In [ ]:
import os
import pandas as pd
from pathlib import Path
# Base configuration template
base_config_five_gene = {
    'n_cells': 6000,
    'simulation_time_before_division': 1000,
    'twin_simulation_time_after_division': 24,
    'twin_measurement_resolution': 1,
    "path_to_connectivity_matrix": f"{path_to_code_repo}/simulation_example_input_data/connectivity_matrix_5_gene_linear_cascade.txt",  # Will be updated per network type
    "param_csv":f"{path_to_code_repo}/simulation_example_input_data/median_parameter.csv",  # Will be updated per network type
    "rows_to_use": [[0,0,0,0,0]],
    "path_to_plot_data": path_to_plot_data,
    "log_file": None,  # Will be updated per network type
    "type": None,  # Will be updated per network type
    "number_of_parallel_parameters": 1,
    "number_of_cores_per_parameter": 18,
}
path_to_simulation_file = f"/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/median_simulation/figure_3_simulations/df_rows_0_0_0_0_0_03122025_221647_ncells_6000_Five_gene_cascade_fixed_K_0_7_6a281f3f.csv"


In [ ]:
twinfer_kwargs = {
    "path_to_simulation_file": path_to_simulation_file,
    "base_config": base_config_five_gene,
    "t1": 1,  #time [hours] after division when t1 sample is collected
    "t2": 20, #time [hours] after division when t2 sample is collected
    "check_for_steady_state": True,
    "threshold_gene_gene_corr": 0.04, #Use direct threshold (used ONLY if use_scramble is False)
    "use_scramble": True, #If set to true, set the p_val_threshold_scrambled_gene_correlation;threshold_gene_gene_corr will NOT be used
    "p_val_threshold_scrambled_gene_correlation": 0.02, #used ONLY if use_scramble is True
    "show_scrambled_distribution_gene_correlation": True, 
    "z_score_threshold_two_states": 10, #Z-score threshold to detect multi-states in the system
    "p_value_threshold_cross_correlation": 0.02,
    "plot_correlation_matrices_as_heatmap": True,
    "have_any_output": True,
    "seed": 101010,
    "infer_direction_for_which_edges": "all-potential-edges", #can be either single-state, all-potential-regulation (gene correlation is significant) or all-edges,
    "merge_time_points": True, #If True, then cells from the two timepoints will be used to calculate gene correlations and random-pair difference correlations
    "n_cores": 16
}

In [ ]:
# Process each network type
all_correlation_matrices = {}
   
# Use the first CSV file found (or you can add logic to select specific one)
print(f"Using simulation file: {os.path.basename(path_to_simulation_file)}")
    
# Update config for this network type
config = base_config_five_gene
network_type = "five_gene_linear_cascade" 
config["type"] = {network_type}

def make_json_safe(obj):
    if hasattr(obj, "to_dict"):      # pandas DataFrame / Series
        return obj.to_dict()
    if isinstance(obj, set):
        return list(obj)
    return obj
    
# Check if required files exist
if not os.path.exists(config["path_to_connectivity_matrix"]):
    print(f"Warning: Connectivity matrix not found for {network_type}")

if not os.path.exists(config["param_csv"]):
    print(f"Warning: Parameter CSV not found for {network_type}")
    
try:
    # Run inference for this network type
    correlation_matrices = infer_with_twinfer(
           **twinfer_kwargs
        )
        
    # Store the correlation matrices
    all_correlation_matrices[network_type] = correlation_matrices
        
    # Save the directional correlation matrix for this network type
    json_safe = {
        k: make_json_safe(v)
        for k, v in correlation_matrices.items()
    }
    path_to_json_file = f"{path_to_plot_data}five_gene_linear_cascade_correlation_matrices.json"

    with open(path_to_json_file, "w") as f:
        json.dump(json_safe, f, indent=2)

    print(correlation_matrices.keys())
    correlation_file_name = f"{path_to_plot_data}/filtered_directional_correlation_type_{network_type}.csv"
    gene_correlation_file_name = f"{path_to_plot_data}/gene_correlation_type_{network_type}.csv"
    correlation_matrices['unfiltered_direction_matrix'].to_csv(correlation_file_name)
    correlation_matrices['pairwise_gene_gene_correlation_matrix'].to_csv(gene_correlation_file_name)
    correlation_matrices
    print(f"Successfully processed {network_type}")
    print(f"Saved correlation matrix to: {correlation_file_name}")
except Exception as e:
    print(f"Error processing {network_type}: {str(e)}")

### Random cross-correlation

In [ ]:
twinfer_kwargs_random = {
    "path_to_simulation_file": path_to_simulation_file,
    "base_config": base_config_five_gene,
    "t1": 1,  #time [hours] after division when t1 sample is collected
    "t2": 20, #time [hours] after division when t2 sample is collected
    "check_for_steady_state": True,
    "threshold_gene_gene_corr": 0.04, #Use direct threshold (used ONLY if use_scramble is False)
    "use_scramble": True, #If set to true, set the p_val_threshold_scrambled_gene_correlation;threshold_gene_gene_corr will NOT be used
    "p_val_threshold_scrambled_gene_correlation": 0.02, #used ONLY if use_scramble is True
    "show_scrambled_distribution_gene_correlation": True, 
    "z_score_threshold_two_states": 10, #Z-score threshold to detect multi-states in the system
    "p_value_threshold_cross_correlation": 0.02,
    "plot_correlation_matrices_as_heatmap": True,
    "have_any_output": True,
    "seed": 101010,
    "infer_direction_for_which_edges": "all-potential-regulation", #can be either single-state, all-potential-regulation (gene correlation is significant) or all-edges,
    "merge_time_points": True, #If True, then cells from the two timepoints will be used to calculate gene correlations and random-pair difference correlations
    "n_cores": 16,
    "remove_twin_structure": True
}

In [ ]:
# Process each network type
all_correlation_matrices = {}
   
# Use the first CSV file found (or you can add logic to select specific one)
print(f"Using simulation file: {os.path.basename(path_to_simulation_file)}")
    
# Update config for this network type
config = base_config_five_gene
network_type = "five_gene_linear_cascade" 
config["type"] = {network_type}

def make_json_safe(obj):
    if hasattr(obj, "to_dict"):      # pandas DataFrame / Series
        return obj.to_dict()
    if isinstance(obj, set):
        return list(obj)
    return obj
    
# Check if required files exist
if not os.path.exists(config["path_to_connectivity_matrix"]):
    print(f"Warning: Connectivity matrix not found for {network_type}")

if not os.path.exists(config["param_csv"]):
    print(f"Warning: Parameter CSV not found for {network_type}")
    
try:
    # Run inference for this network type
    correlation_matrices = infer_with_twinfer(
           **twinfer_kwargs_random
        )
        
    # Store the correlation matrices
    all_correlation_matrices[network_type] = correlation_matrices
        
    # Save the directional correlation matrix for this network type
    json_safe = {
        k: make_json_safe(v)
        for k, v in correlation_matrices.items()
    }
    path_to_json_file = f"{path_to_plot_data}five_gene_linear_cascade_correlation_matrices_random.json"

    with open(path_to_json_file, "w") as f:
        json.dump(json_safe, f, indent=2)

    print(correlation_matrices.keys())
    correlation_file_name = f"{path_to_plot_data}/filtered_directional_correlation_type_{network_type}_random.csv"
    gene_correlation_file_name = f"{path_to_plot_data}/gene_correlation_type_{network_type}_random.csv"
    correlation_matrices['unfiltered_direction_matrix'].to_csv(correlation_file_name)
    correlation_matrices['pairwise_gene_gene_correlation_matrix'].to_csv(gene_correlation_file_name)
    correlation_matrices
    print(f"Successfully processed {network_type}")
    print(f"Saved correlation matrix to: {correlation_file_name}")
except Exception as e:
    print(f"Error processing {network_type}: {str(e)}")